---
title: Sharing compute artifacts
---

Every parameterised phasic graph has a deterministic content hash (`ptd_graph_content_hash`). The first time you call `g.expectation()` (or `g.pdf()` / `g.moments()`), the C side runs the symbolic elimination and writes the result to `~/.phasic_cache/parameterized_reward_compute/<hash>.bin`. The next call on a structurally identical graph reuses that file — even in a different process.

The sharing system lets you exchange those `.bin` files with other users via a public GitHub registry at [`munch-group/phasic-traces`](https://github.com/munch-group/phasic-traces). Two methods on `Graph` cover the whole story:

If someone published a `.bin` for this graph, you download it into the local cache using `g.pull_cache()`. To list models available at `phasic-traces` use the top-level function `list_computes` for browsing without a graph. E.g., `phasic.list_computes(domain='population-genetics')`

To make an elimination available to others, use `g.push_cache(id=..., description=...)` to compute it (unless already cached locally) and submit it as a pull request for the `phasic-traces` repository. This feature needs `git` and the [GitHub CLI](https://cli.github.com/) `gh`, authenticated once with `gh auth login` in a terminal.Both are easily installed using either Pixi, Conda or Pip:

::: {.panel-tabset}

## Pixi

```bash
pixi global install git gh
```

## Conda

```bash
conda install git gh
```

## Pip

```bash
pip install git gh
```

:::

## Setup

In [1]:
import json, tempfile, shutil, hashlib
from pathlib import Path
import numpy as np

import phasic
from phasic.exceptions import PTDBackendError, PTDFormatError

## Browsing the registry

`phasic.list_computes()` reads the latest `registry.json` from the public GitHub repo and returns one dict per published artifact. Filter by `domain`, `model_type`, or `tags` to narrow the list.

In [2]:
# In a connected environment this fetches the live registry from
# github.com/munch-group/phasic-traces.
#
# Uncomment the next two lines to try it:
# for e in phasic.list_computes():
#     print(f"{e['compute_id']:25s}  vertices={e.get('vertices', '?')}")

print('(network fetch skipped in tutorial run)')

(network fetch skipped in tutorial run)


## Pull: reusing someone else's elimination

When you build a parameterised graph that's structurally identical to one that someone has published, `g.pull_cache()` downloads the pre-computed `.bin` so the next `g.expectation()` skips the O(n^3) elimination.

To make this tutorial self-contained we set up a tiny local registry and tell phasic to read from a `file://` URL instead of GitHub. In real use you just call `g.pull_cache()` — no setup.

In [3]:
demo_root  = Path(tempfile.mkdtemp(prefix='phasic_share_demo_'))
fake_origin = demo_root / 'origin'; fake_origin.mkdir()
reg_cache   = demo_root / 'registry_cache'; reg_cache.mkdir()

# 1. Build a small parameterised graph and run elimination locally.
#    This populates ~/.phasic_cache/parameterized_reward_compute/<hash>.bin.
g_publisher = phasic.Graph(1)
v0 = g_publisher.starting_vertex()
v1 = g_publisher.find_or_create_vertex([1])
v2 = g_publisher.find_or_create_vertex([2])
v0.add_edge(v1, [1.0])
v1.add_edge(v2, [1.0])
g_publisher.update_weights([2.0])
print(f'publisher expectation: {g_publisher.expectation()}')

# 2. The graph's content hash is what identifies the artifact.
hash_hex = phasic.hash.compute_graph_hash(g_publisher).hash_hex
print(f'graph hash: {hash_hex[:24]}...')

publisher expectation: 0.5
graph hash: 29dc136099b6cf8de1d9180a...


In [4]:
# 3. Stage the .bin in our fake-origin directory (simulates a
#    publish). In a real publish, push_cache() does this AND opens
#    a PR for you.
from phasic.compute_repository import _param_compute_cache_dir
local_bin = _param_compute_cache_dir() / f'{hash_hex}.bin'
published = fake_origin / f'{hash_hex}.bin'
shutil.copy2(local_bin, published)
sha = hashlib.sha256(published.read_bytes()).hexdigest()
print(f'staged: {published.name} ({published.stat().st_size:,} bytes)')
print(f'sha256: {sha[:32]}...')

# 4. Hand-write a minimal registry.json pointing at the file.
registry_data = {
    'version': '2.0', 'format': 'ptd_pcg',
    'computes': {
        'demo_chain': {
            'graph_hash':      hash_hex,
            'format_revision': 2,
            'artifacts': {
                'parent': {
                    'cid_or_path': f'file://{published}',
                    'sha256':      sha,
                    'size_bytes':  published.stat().st_size,
                },
                'scc': [],
            },
            'metadata': {
                'description': 'Two-step parameterised chain (demo)',
                'domain':      'demo',
                'model_type':  'chain',
                'vertices':    g_publisher.vertices_length(),
                'param_length': 1,
                'tags':        ['demo', 'tutorial'],
            },
        },
    },
}
(reg_cache / 'registry.json').write_text(json.dumps(registry_data))

staged: 29dc136099b6cf8de1d9180a234a2358b63c5ddca6649889f9ed6a25d914418c.bin (2,304 bytes)
sha256: 7077b684810fc7f8de111dcde1b3175c...


659

In [5]:
# 5. Tell phasic to read the registry from our local mock.
from phasic.compute_repository import ComputeRegistry, _get_default_registry
import phasic.compute_repository as cr

# Reset the module-level default so it picks up our mock.
cr._default_registry = ComputeRegistry(
    registry_repo='demo/demo', cache_dir=reg_cache, auto_update=False)

# Sanity: list what's available.
for e in phasic.list_computes():
    print(f"{e['compute_id']:15s}  hash={e['graph_hash'][:12]}... "
          f"vertices={e.get('vertices', '?')}")

demo_chain       hash=29dc136099b6... vertices=3


Now simulate a fresh consumer: a different process that happens to build the same graph and wants to skip the elimination.

In [6]:
# Clear the local cache to simulate 'fresh machine'.
if local_bin.exists():
    local_bin.unlink()

# Build the structurally identical graph.
g_consumer = phasic.Graph(1)
w0 = g_consumer.starting_vertex()
w1 = g_consumer.find_or_create_vertex([1])
w2 = g_consumer.find_or_create_vertex([2])
w0.add_edge(w1, [1.0])
w1.add_edge(w2, [1.0])
g_consumer.update_weights([2.0])

# The whole sharing UI:
hit = g_consumer.pull_cache()
print(f'pull_cache returned: {hit}')
print(f'cache file present: {local_bin.exists()}')

# The next call uses the downloaded elimination.
print(f'consumer expectation: {g_consumer.expectation()}')

pull_cache returned: True
cache file present: True
consumer expectation: 0.5


If no one has published the graph, `pull_cache()` returns `False` and the cache is untouched.

In [7]:
# A different graph that isn't in the registry.
g_other = phasic.Graph(1)
x0 = g_other.starting_vertex()
x1 = g_other.find_or_create_vertex([1])
x2 = g_other.find_or_create_vertex([2])
x3 = g_other.find_or_create_vertex([3])
x0.add_edge(x1, [1.0])
x1.add_edge(x2, [1.0])
x2.add_edge(x3, [1.0])
x3.add_edge(x0, [0.0])  # dummy parameter slot
g_other.update_weights([2.0])

hit = g_other.pull_cache()
print(f'pull_cache returned: {hit}')   # False — no entry for this hash

pull_cache returned: False


## Push: publishing your own elimination

`g.push_cache(id=..., description=...)` packages the local `.bin`, opens a pull request against `phasic-traces`, and returns the PR URL. The maintainer reviews and merges it; from that moment, anyone who calls `pull_cache()` on the same graph gets your published artifact.

**Prerequisite (one-time):** install [`gh`](https://cli.github.com/) and run `gh auth login` in a terminal. `push_cache()` does not prompt inside the notebook; it just raises a clear `PTDBackendError` if `gh` is missing or unauthenticated.

Use `dry_run=True` to inspect what the published entry would look like without actually opening a PR. This works offline.

In [8]:
entry_json = g_publisher.push_cache(
    id='demo_chain_v1',
    description='Two-step parameterised chain (demo)',
    domain='demo',
    model_type='chain',
    tags=['demo', 'tutorial'],
    dry_run=True,
)
print(entry_json)

{
  "demo_chain_v1": {
    "graph_hash": "29dc136099b6cf8de1d9180a234a2358b63c5ddca6649889f9ed6a25d914418c",
    "format_revision": 2,
    "artifacts": {
      "parent": {
        "cid_or_path": "artifacts/29dc136099b6cf8de1d9180a234a2358b63c5ddca6649889f9ed6a25d914418c.bin",
        "sha256": "27a7021dba42413895f99f633c7aadee75b5c95c0b1c7c6bb588f00b815cdbb3",
        "size_bytes": 2304
      },
      "scc": [
        {
          "cid_or_path": "artifacts/scc_6a6d87a7e303d7fb422f4eece14a59aa03a8cdaaaa5d374cd71096944a948486.bin",
          "scc_hash": "6a6d87a7e303d7fb422f4eece14a59aa03a8cdaaaa5d374cd71096944a948486",
          "sha256": "b487cef8fb806dd41404d448ebfd3a101543bd6fdee836d999d27c014f69d7ed",
          "size_bytes": 8056
        },
        {
          "cid_or_path": "artifacts/scc_ddc3218c9ae13954225efba2a3f16068214f18901364b150fb4bea51c9d4708a.bin",
          "scc_hash": "ddc3218c9ae13954225efba2a3f16068214f18901364b150fb4bea51c9d4708a",
          "sha256": "b8e9372a0aa80c2

To open the PR for real, drop `dry_run=True`:

``python
pr_url = g_publisher.push_cache(
    id='demo_chain_v1',
    description='Two-step parameterised chain (demo)',
    domain='demo',
    model_type='chain',
    tags=['demo', 'tutorial'],
)
print(pr_url)   # https://github.com/munch-group/phasic-traces/pull/N
``

The CLI workflow:

1. `push_cache` clones `phasic-traces` to a temporary directory.
2. Saves the `.bin` to `artifacts/<hash>.bin` and any per-SCC files.
3. Splices a new entry into `registry.json`.
4. Commits to a feature branch `phasic-publish/<id>`.
5. Pushes the branch to your fork.
6. Runs `gh pr create` and prints the PR URL.

All of this is bookkeeping — the artifact itself is the `.bin` you can also share with collaborators by other means (email, Slack, etc.) for ad-hoc use.

## What happens during `expectation()`

After `pull_cache()` has populated the local cache, the next `g.expectation()` call goes through the normal C path:

1. The C side computes `ptd_graph_content_hash(graph)`.
2. It looks for `~/.phasic_cache/parameterized_reward_compute/<hash>.bin`.
3. If present, it loads the symbolic elimination from the file and applies the current `theta` to get a concrete result.
4. If absent, it runs the O(n^3) elimination and writes a new `.bin`.

`pull_cache()` just ensures step 2 succeeds. It does not load the compute graph into memory itself; the C side does that on the next call.

## When to publish

Publish when:

- The elimination is expensive (seconds to minutes) and several people in your group will run the same model.
- You want to make a published-paper model trivially reproducible — anyone with phasic installed can verify your results without re-running the eliminations.
- You're building a workflow that runs on many machines (CI, clusters) and you don't want every machine to re-eliminate.

Don't publish:

- Toy or one-off graphs.
- Models whose structure you're still iterating on. The hash depends on topology + coefficients, so any structural change creates a new hash anyway.

## Summary

| Task | API |
|------|-----|
| Browse registry | `phasic.list_computes(domain=..., model_type=..., tags=...)` |
| Download elimination | `g.pull_cache()` — returns `True`/`False` |
| Force re-download | `g.pull_cache(force=True)` |
| Publish elimination | `g.push_cache(id=..., description=...)` |
| Inspect before publishing | `g.push_cache(..., dry_run=True)` |
| One-time auth | `gh auth login` in a terminal |

In [9]:
# Cleanup
shutil.rmtree(demo_root, ignore_errors=True)
if local_bin.exists():
    local_bin.unlink()
# Restore the default registry so subsequent cells aren't affected
cr._default_registry = None
print('done')

done
